In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, time, json, glob, random
import numpy as np
import pandas as pd

Mounted at /content/drive


In [ ]:
# =========================
# SECTION 1: SETUP & CONFIG
# =========================

DATA_DIR  = "/content/drive/MyDrive/1149108/RiGAT-CMI/Dataset/CMI-9905"
RiNALMo_DIR = f"{DATA_DIR}/rinalmo_mega"                  # change 'mega' or 'micro'
FOLD_ROOT = f"{DATA_DIR}/5fold_CV"
INDEP_DIR = f"{DATA_DIR}/independent_test"

OUT_DIR    = f"{DATA_DIR}/outputs_gat_RiNALMo_Mega"           # CSVs
CKPT_ROOT  = f"{DATA_DIR}/checkpoints_gat_RiNALMo_Mega"       # checkpoints /fold*/ /final/
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CKPT_ROOT, exist_ok=True)

MI_EMB_PATH = f"{RiNALMo_DIR}/miRNA_rinalmo_mega.npy"         # 'giga' or 'mega' or 'micro'
CI_EMB_PATH = f"{RiNALMo_DIR}/circRNA_rinalmo_mega.npy"

CFG = dict(
    seed=42,
    folds=5,
    # GAT
    hidden_dim=256,
    layers=3,
    heads=4,
    dropout=0.20,
    message_dropout=0.20,
    residual=True,
    # Optim
    lr=1e-4,
    weight_decay=1e-5,
    epochs=100,
    batch_size=4096,
    # Node-dropout
    keep_node=0.80,
    # Early stopping
    patience=10,
    val_split=0.10,
    device="cuda" if __import__("torch").cuda.is_available() else "cpu",
)

def set_seed(seed=42):
    import torch
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def ts():
    import time
    return time.strftime("%Y%m%d-%H%M%S")

set_seed(CFG["seed"])
print("Paths\n- FOLD_ROOT:", FOLD_ROOT, "\n- INDEP_DIR:", INDEP_DIR, "\n- CKPT_ROOT:", CKPT_ROOT)
print("Device:", CFG["device"])


Paths
- FOLD_ROOT: /content/drive/MyDrive/1149108/DGCLCMI-RiNALMo-new/Dataset/CMI-9905/5fold_CV 
- INDEP_DIR: /content/drive/MyDrive/1149108/DGCLCMI-RiNALMo-new/Dataset/CMI-9905/independent_test 
- CKPT_ROOT: /content/drive/MyDrive/1149108/DGCLCMI-RiNALMo-new/Dataset/CMI-9905/checkpoints_gat_RiNALMo_Mega
Device: cpu


In [ ]:
# =========================
# SECTION 2: LOAD DATA 
# =========================
import torch, os, glob

Xm = np.load(MI_EMB_PATH); Xc = np.load(CI_EMB_PATH)
n_mi, H = Xm.shape; n_ci = Xc.shape[0]
device = torch.device(CFG["device"])

E_mi = torch.from_numpy(Xm).float().to(device)
E_ci = torch.from_numpy(Xc).float().to(device)
X = torch.cat([E_mi, E_ci], dim=0)
print("Nodes:", n_mi, n_ci, "| Feature dim:", H)

def read_edges_2col(path):
    return pd.read_csv(path, header=None, sep='\t', engine='python').iloc[:, :2].to_numpy(dtype=np.int64)

# Independent test
ind_pos = read_edges_2col(os.path.join(INDEP_DIR, "pos.csv"))
ind_neg = read_edges_2col(os.path.join(INDEP_DIR, "neg.csv"))
ind_pairs_np = np.vstack([
    np.column_stack([ind_pos[:,0], ind_pos[:,1], np.ones(len(ind_pos))]),
    np.column_stack([ind_neg[:,0], ind_neg[:,1], np.zeros(len(ind_neg))]),
]).astype(np.int64)

# fold*
fold_dirs = sorted([d for d in glob.glob(os.path.join(FOLD_ROOT, "fold*")) if os.path.isdir(d)])
print("Folds:", [os.path.basename(d) for d in fold_dirs])


Nodes: 962 2346 | Feature dim: 640
Folds: ['fold0', 'fold1', 'fold2', 'fold3', 'fold4']


In [5]:
# =========================
# SECTION 3: MODEL & HELPERS
# =========================
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, average_precision_score, matthews_corrcoef, confusion_matrix

def edges_to_coo(n_mi, n_ci, pos_edges):
    src = pos_edges[:,0]
    dst = pos_edges[:,1] + n_mi
    row = np.concatenate([src, dst])
    col = np.concatenate([dst, src])
    idx = np.stack([row, col], axis=0)
    val = np.ones(idx.shape[1], dtype=np.float32)
    return torch.from_numpy(idx).long(), torch.from_numpy(val).float()

def apply_node_dropout(indices, values, keep_prob, num_nodes, device):
    if keep_prob >= 1.0: return indices, values
    g = torch.Generator(device=device); g.manual_seed(1234)
    mask = (torch.rand(num_nodes, generator=g, device=device) < keep_prob).float()
    keep_src = mask[indices[0]]; keep_dst = mask[indices[1]]
    keep_edge = (keep_src * keep_dst) > 0.5
    return indices[:, keep_edge], values[keep_edge]

def normalize_adj(indices, values, num_nodes, device, add_self_loops=True):
    i, v = indices, values
    if add_self_loops:
        sl = torch.arange(num_nodes, device=device)
        self_i = torch.stack([sl, sl], dim=0)
        i = torch.cat([i, self_i], dim=1)
        v = torch.cat([v, torch.ones(num_nodes, device=device)], dim=0)
    deg = torch.zeros(num_nodes, device=device).scatter_add_(0, i[0], v)
    deg = torch.clamp(deg, min=1e-12)
    d_inv_sqrt = torch.pow(deg, -0.5)
    norm_v = d_inv_sqrt[i[0]] * v * d_inv_sqrt[i[1]]
    return i, norm_v

def compute_metrics(y_true, y_score, thr=0.5):
    auc = roc_auc_score(y_true, y_score)
    aupr = average_precision_score(y_true, y_score)
    y_pred = (y_score >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    acc  = (tp+tn)/max(tn+fp+fn+tp,1)
    sens = tp/max(tp+fn,1)
    spec = tn/max(tn+fp,1)
    mcc  = matthews_corrcoef(y_true, y_pred) if (tp+tn+fp+fn)>0 else 0.0
    return dict(AUC=auc, AUPR=aupr, ACC=acc, SENS=sens, SPEC=spec, MCC=mcc)

class SimpleGATLayer(nn.Module):
    def __init__(self, in_dim, out_dim, heads=4, dropout=0.0, negative_slope=0.2):
        super().__init__()
        self.in_dim, self.out_dim, self.heads = in_dim, out_dim, heads
        self.dropout = nn.Dropout(dropout) if dropout>0 else nn.Identity()
        self.leakyrelu = nn.LeakyReLU(negative_slope)
        self.W = nn.Parameter(torch.Tensor(heads, in_dim, out_dim))
        self.a_src = nn.Parameter(torch.Tensor(heads, out_dim))
        self.a_dst = nn.Parameter(torch.Tensor(heads, out_dim))
        self.reset_parameters()

    def reset_parameters(self):
        for h in range(self.heads):
            nn.init.xavier_uniform_(self.W[h])
        nn.init.xavier_uniform_(self.a_src.unsqueeze(0))
        nn.init.xavier_uniform_(self.a_dst.unsqueeze(0))

    def forward(self, x, edge_index, message_dropout=0.0):
        N = x.size(0)
        src, dst = edge_index[0], edge_index[1]
        xW = torch.einsum('nf,hfo->hno', x, self.W)  # (h,N,d)
        x_i = xW[:, src, :] ; x_j = xW[:, dst, :]
        e = self.leakyrelu(
            torch.einsum('hkd,hd->hk', x_i, self.a_src) +
            torch.einsum('hkd,hd->hk', x_j, self.a_dst)
        )  # (h,E)

        attn = []
        use_scatter = hasattr(torch.Tensor, 'scatter_reduce_')
        for h in range(self.heads):
            s = e[h]
            if use_scatter:
                mx = torch.full((N,), -1e9, device=x.device)
                mx.scatter_reduce_(0, dst, s, reduce='amax', include_self=True)
                exp_s = torch.exp(s - mx[dst])
                den = torch.zeros(N, device=x.device)
                den.scatter_reduce_(0, dst, exp_s, reduce='sum', include_self=True)
                alpha = exp_s / (den[dst] + 1e-12)
            else:
                exp_s = torch.exp(s)
                den = torch.zeros(N, device=x.device)
                den.index_add_(0, dst, exp_s)
                alpha = exp_s / (den[dst] + 1e-12)
            if message_dropout>0 and self.training:
                alpha = F.dropout(alpha, p=message_dropout, training=True)
            attn.append(alpha)
        attn = torch.stack(attn, dim=0)

        out = torch.zeros(self.heads, N, self.out_dim, device=x.device)
        for h in range(self.heads):
            msg = x_i[h] * attn[h].unsqueeze(-1)
            out[h].index_add_(0, dst, msg)
        out = out.permute(1,0,2).reshape(N, self.heads*self.out_dim)
        return self.dropout(out)

class GATEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim=256, layers=2, heads=4, dropout=0.0, message_dropout=0.0, residual=True):
        super().__init__()
        self.layers = nn.ModuleList()
        self.residual = residual
        self.message_dropout = message_dropout
        in_dims = [in_dim] + [heads*hidden_dim] * (layers - 1)
        for l in range(layers):
            self.layers.append(SimpleGATLayer(in_dims[l], hidden_dim, heads=heads, dropout=dropout))
        self.act = nn.ELU()

    def forward(self, x, edge_index):
        h = x; outs=[]
        for gat in self.layers:
            h_new = gat(h, edge_index, message_dropout=self.message_dropout)
            h_new = self.act(h_new)
            h = h + h_new if (self.residual and h_new.shape==h.shape) else h_new
            outs.append(h)
        return torch.cat(outs, dim=-1)

class GATLinkPredictor(nn.Module):
    def __init__(self, in_dim, hidden_dim=256, layers=2, heads=4, dropout=0.0, message_dropout=0.0, residual=True):
        super().__init__()
        self.encoder = GATEncoder(in_dim, hidden_dim, layers, heads, dropout, message_dropout, residual)
        out_dim = layers*heads*hidden_dim
        self.scorer = nn.Bilinear(out_dim, out_dim, 1, bias=True)

    def forward(self, X, edge_index, pairs):
        z = self.encoder(X, edge_index)
        left  = z[pairs[:,0]]; right = z[pairs[:,1]]
        return self.scorer(left, right).squeeze(-1)


In [ ]:
# =========================
# SECTION 4: 5-FOLD CV 
# =========================
def build_edge_index_from_pos(pos_edges, keep_node):
    idx, val = edges_to_coo(n_mi, n_ci, pos_edges)
    idx = idx.to(device); val = val.to(device)
    idx, val = apply_node_dropout(idx, val, keep_prob=CFG["keep_node"], num_nodes=n_mi+n_ci, device=device)
    idx, val = normalize_adj(idx, val, num_nodes=n_mi+n_ci, device=device, add_self_loops=True)
    return idx

def make_pairs(mi_np, ci_np):
    a = torch.from_numpy(mi_np.astype(np.int64)).to(device)
    b = torch.from_numpy((ci_np + n_mi).astype(np.int64)).to(device)
    return torch.stack([a, b], dim=1)

def iterate_batches(pairs, labels, bs):
    N = pairs.size(0)
    for s in range(0, N, bs):
        e = min(s+bs, N)
        yield pairs[s:e], labels[s:e]

def eval_phase(model, edge_index, pairs_t, y_t):
    model.eval()
    with torch.no_grad():
        logits = []
        for p,_ in iterate_batches(pairs_t, y_t, CFG["batch_size"]):
            logits.append(model(X, edge_index, p))
        scores = torch.sigmoid(torch.cat(logits)).detach().cpu().numpy()
        true = y_t.detach().cpu().numpy()
        return compute_metrics(true, scores)

log_rows = []
best_rows = []   # saved best-per-fold

for fold_id, fold_dir in enumerate(fold_dirs):
    fname = os.path.basename(fold_dir)  # fold0..fold4
    print(f"\n=== CV {fname} ===")
    CKPT_DIR = os.path.join(CKPT_ROOT, fname)
    os.makedirs(CKPT_DIR, exist_ok=True)


    pos_tr = read_edges_2col(os.path.join(fold_dir, "pos_train.csv"))
    pos_te = read_edges_2col(os.path.join(fold_dir, "pos_test.csv"))
    neg_tr = read_edges_2col(os.path.join(fold_dir, "neg_train.csv"))
    neg_te = read_edges_2col(os.path.join(fold_dir, "neg_test.csv"))

    # graph from train positives of fold
    edge_index = build_edge_index_from_pos(pos_tr, keep_node=CFG["keep_node"])

    # train set + val split
    tr_np = np.vstack([
        np.column_stack([pos_tr[:,0], pos_tr[:,1], np.ones(len(pos_tr))]),
        np.column_stack([neg_tr[:,0], neg_tr[:,1], np.zeros(len(neg_tr))]),
    ]).astype(np.int64)
    rng = np.random.default_rng(CFG["seed"] + 7 + fold_id)
    rng.shuffle(tr_np)
    n_val = max(1, int(CFG["val_split"]*len(tr_np))); n_val = min(n_val, len(tr_np)-1)
    val_np = tr_np[:n_val]; tr_np = tr_np[n_val:]

    tr_pairs = make_pairs(tr_np[:,0], tr_np[:,1]); tr_y = torch.from_numpy(tr_np[:,2].astype(np.float32)).to(device)
    val_pairs = make_pairs(val_np[:,0], val_np[:,1]); val_y = torch.from_numpy(val_np[:,2].astype(np.float32)).to(device)

    te_np = np.vstack([
        np.column_stack([pos_te[:,0], pos_te[:,1], np.ones(len(pos_te))]),
        np.column_stack([neg_te[:,0], neg_te[:,1], np.zeros(len(neg_te))]),
    ]).astype(np.int64)
    te_pairs = make_pairs(te_np[:,0], te_np[:,1]); te_y = torch.from_numpy(te_np[:,2].astype(np.float32)).to(device)

    # model fold
    model = GATLinkPredictor(
        in_dim=X.shape[1],
        hidden_dim=CFG["hidden_dim"],
        layers=CFG["layers"],
        heads=CFG["heads"],
        dropout=CFG["dropout"],
        message_dropout=CFG["message_dropout"],
        residual=CFG["residual"],
    ).to(device)
    optim = torch.optim.Adam(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])

    best_val, best_state, best_ep = -1.0, None, None
    no_imp = 0
    best_path = os.path.join(CKPT_DIR, "best_fold.pt")

    for ep in range(1, CFG["epochs"]+1):
        # ---- train ----
        model.train()
        perm = torch.randperm(tr_pairs.size(0), device=device)
        tr_pairs = tr_pairs[perm]; tr_y = tr_y[perm]
        for p,y in iterate_batches(tr_pairs, tr_y, CFG["batch_size"]):
            optim.zero_grad(); logits = model(X, edge_index, p)
            loss = F.binary_cross_entropy_with_logits(logits, y)
            loss.backward(); optim.step()

        # ---- eval: val & fold_test ----
        val_m = eval_phase(model, edge_index, val_pairs, val_y)
        te_m  = eval_phase(model, edge_index, te_pairs, te_y)

        def r4(x): return float(pd.Series([x]).round(4).iloc[0])

        # per-epoch log
        log_rows.append({"time": ts(), "fold": fname, "epoch": ep, "phase": "val",
                        **{k: r4(v) for k,v in val_m.items()}})
        log_rows.append({"time": ts(), "fold": fname, "epoch": ep, "phase": "fold_test",
                        **{k: r4(v) for k,v in te_m.items()}})
        print(f"[{fname}] epoch {ep:03d} | val AUC={val_m['AUC']:.4f} | test AUC={te_m['AUC']:.4f}")

        #save checkpoint per epoch
        ep_path = os.path.join(CKPT_DIR, f"epoch_{ep:03d}.pt")
        torch.save({"model": model.state_dict(), "cfg": CFG, "epoch": ep}, ep_path)

        #best-by-val per fold
        if val_m["AUC"] > best_val:
            best_val = val_m["AUC"]
            best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            best_ep = ep
            torch.save({"model": best_state, "cfg": CFG, "epoch": best_ep, "best_val_auc": float(r4(best_val))}, best_path)
            no_imp = 0
        else:
            no_imp += 1

        if no_imp >= CFG["patience"]:
            print(f"[{fname}] early stop at epoch {ep} (best epoch {best_ep}, best val AUC={best_val:.4f})")
            break

    #last-of-fold
    last_path = os.path.join(CKPT_DIR, "last_fold.pt")
    torch.save({"model": model.state_dict(), "cfg": CFG, "epoch": ep}, last_path)

    # evaluated fold_test (best-per-fold)
    if best_state is not None:
        model.load_state_dict(best_state)
        te_m_best = eval_phase(model, edge_index, te_pairs, te_y)
        best_rows.append({
            "fold": fname, "epoch": best_ep, "phase": "fold_test_best",
            **{k: float(pd.Series([v]).round(4).iloc[0]) for k,v in te_m_best.items()}
        })


In [ ]:
# =========================
# SECTION 5: Independent & Final Model
# =========================
df = pd.DataFrame(log_rows)
for c in ["AUC","AUPR","ACC","SENS","SPEC","MCC"]:
    if c in df.columns: df[c] = df[c].astype(float).round(4)

# mean per fold (per epoch) & mean across folds (CV)
fold_means = (df.groupby(["fold","phase"], as_index=False)[["AUC","AUPR","ACC","SENS","SPEC","MCC"]]
                .mean().round(4))
fold_means["epoch"] = "mean_epochs"; fold_means["time"] = ts()

overall_means = (fold_means.groupby(["phase"], as_index=False)[["AUC","AUPR","ACC","SENS","SPEC","MCC"]]
                   .mean().round(4))
overall_means["fold"]  = "mean_5folds"; overall_means["epoch"] = "mean_epochs"; overall_means["time"] = ts()

cv_df_full = pd.concat([df, fold_means, overall_means], ignore_index=True)

def make_pairs(mi_np, ci_np):
    a = torch.from_numpy(mi_np.astype(np.int64)).to(device)
    b = torch.from_numpy((ci_np + n_mi).astype(np.int64)).to(device)
    return torch.stack([a, b], dim=1)

def build_edge_index_from_pos(pos_edges, keep_node):
    idx, val = edges_to_coo(n_mi, n_ci, pos_edges)
    idx = idx.to(device); val = val.to(device)
    idx, val = apply_node_dropout(idx, val, keep_prob=CFG["keep_node"], num_nodes=n_mi+n_ci, device=device)
    idx, val = normalize_adj(idx, val, num_nodes=n_mi+n_ci, device=device, add_self_loops=True)
    return idx

#  train80 from fold
all_pos_train = []
all_neg_train = []
for fold_dir in fold_dirs:
    all_pos_train.append(read_edges_2col(os.path.join(fold_dir, "pos_train.csv")))
    all_neg_train.append(read_edges_2col(os.path.join(fold_dir, "neg_train.csv")))
pos_train80_full = np.vstack(all_pos_train)
neg_train80_full = np.vstack(all_neg_train)

edge_index_full = build_edge_index_from_pos(pos_train80_full, keep_node=CFG["keep_node"])

sup_np = np.vstack([
    np.column_stack([pos_train80_full[:,0], pos_train80_full[:,1], np.ones(len(pos_train80_full))]),
    np.column_stack([neg_train80_full[:,0], neg_train80_full[:,1], np.zeros(len(neg_train80_full))]),
]).astype(np.int64)
rng = np.random.default_rng(CFG["seed"] + 999)
rng.shuffle(sup_np)
n_val = max(1, int(CFG["val_split"]*len(sup_np))); n_val = min(n_val, len(sup_np)-1)
val_np_final = sup_np[:n_val]; tr_np_final = sup_np[n_val:]

tr_pairs_f = make_pairs(tr_np_final[:,0], tr_np_final[:,1]); tr_y_f = torch.from_numpy(tr_np_final[:,2].astype(np.float32)).to(device)
val_pairs_f = make_pairs(val_np_final[:,0], val_np_final[:,1]); val_y_f = torch.from_numpy(val_np_final[:,2].astype(np.float32)).to(device)

# independent tensors
ind_pairs_t = make_pairs(ind_pairs_np[:,0], ind_pairs_np[:,1])
ind_y_t     = torch.from_numpy(ind_pairs_np[:,2].astype(np.float32)).to(device)

FINAL_DIR = os.path.join(CKPT_ROOT, "final")
os.makedirs(FINAL_DIR, exist_ok=True)

final_model = GATLinkPredictor(
    in_dim=X.shape[1],
    hidden_dim=CFG["hidden_dim"],
    layers=CFG["layers"],
    heads=CFG["heads"],
    dropout=CFG["dropout"],
    message_dropout=CFG["message_dropout"],
    residual=CFG["residual"],
).to(device)
final_optim = torch.optim.Adam(final_model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])

best_val_f, best_state_f, best_ep_f = -1.0, None, None
no_imp_f = 0

def eval_phase_simple(model, edge_index, pairs_t, y_t):
    model.eval()
    with torch.no_grad():
        logits = []
        for s in range(0, pairs_t.size(0), CFG["batch_size"]):
            e = min(s+CFG["batch_size"], pairs_t.size(0))
            logits.append(model(X, edge_index, pairs_t[s:e]))
        scores = torch.sigmoid(torch.cat(logits)).detach().cpu().numpy()
        true = y_t.detach().cpu().numpy()
        return compute_metrics(true, scores)

print("\n=== FINAL (train80 -> independent) ===")
for ep in range(1, CFG["epochs"]+1):
    final_model.train()
    perm = torch.randperm(tr_pairs_f.size(0), device=device)
    tr_pairs_f = tr_pairs_f[perm]; tr_y_f = tr_y_f[perm]
    for s in range(0, tr_pairs_f.size(0), CFG["batch_size"]):
        e = min(s+CFG["batch_size"], tr_pairs_f.size(0))
        p = tr_pairs_f[s:e]; y = tr_y_f[s:e]
        final_optim.zero_grad(); logits = final_model(X, edge_index_full, p)
        loss = F.binary_cross_entropy_with_logits(logits, y)
        loss.backward(); final_optim.step()

    val_m_f = eval_phase_simple(final_model, edge_index_full, val_pairs_f, val_y_f)
    print(f"[final] epoch {ep:03d} | val AUC={val_m_f['AUC']:.4f}")

    # ckpt per epoch
    torch.save({"model": final_model.state_dict(), "cfg": CFG, "epoch": ep},
               os.path.join(FINAL_DIR, f"epoch_{ep:03d}.pt"))

    if val_m_f["AUC"] > best_val_f:
        best_val_f = val_m_f["AUC"]; best_ep_f = ep
        best_state_f = {k:v.detach().cpu().clone() for k,v in final_model.state_dict().items()}
        torch.save({"model": best_state_f, "cfg": CFG, "epoch": best_ep_f, "best_val_auc": float(pd.Series([best_val_f]).round(4).iloc[0])},
                   os.path.join(FINAL_DIR, "best_final.pt"))
        no_imp_f = 0
    else:
        no_imp_f += 1
    if no_imp_f >= CFG["patience"]:
        print(f"[final] early stop at epoch {ep} (best epoch {best_ep_f}, best val AUC={best_val_f:.4f})")
        break

# load best_final & evaluate independent
if best_state_f is not None:
    final_model.load_state_dict(best_state_f)

ind_m_final = eval_phase_simple(final_model, edge_index_full, ind_pairs_t, ind_y_t)
ind_row = {"time": ts(), "fold": "final_model", "epoch": "best", "phase": "independent_final",
           **{k: float(pd.Series([v]).round(4).iloc[0]) for k,v in ind_m_final.items()}}
print("Independent FINAL:", ind_row)

best_df = pd.DataFrame(best_rows)  # best-per-fold
for c in ["AUC","AUPR","ACC","SENS","SPEC","MCC"]:
    if c in best_df.columns: best_df[c] = best_df[c].astype(float).round(4)

# mean±std best-per-fold (fold_test_best)
def mean_std_str(x):
    return f"{np.mean(x):.4f}±{np.std(x, ddof=1):.4f}" if len(x)>1 else f"{np.mean(x):.4f}±0.0000"
summary = {}
for m in ["AUC","AUPR","ACC","SENS","SPEC","MCC"]:
    summary[m] = mean_std_str(best_df[m].values) if m in best_df.columns else "NA"
summary_row = {"fold":"mean±std_5folds","epoch":"best","phase":"fold_test_best_summary", **summary}
best_df_summary = pd.concat([best_df, pd.DataFrame([summary_row])], ignore_index=True)

# 4) Save CSVs
csv_log_path     = os.path.join(OUT_DIR, f"metrics_gat_CV_per_epoch_{ts()}.csv")
csv_summary_path = os.path.join(OUT_DIR, f"metrics_gat_CV_best_and_final_{ts()}.csv")

# per-epoch (CV) + mean_epochs + mean_5folds + independent_final
cv_and_final_df = pd.concat([cv_df_full, pd.DataFrame([ind_row])], ignore_index=True)
cv_and_final_df.to_csv(csv_log_path, index=False)
best_df_summary.to_csv(csv_summary_path, index=False)

print("\nSaved:")
print(" - Per-epoch CV + means + independent_final:", csv_log_path)
print(" - Summary best-per-fold (mean±std) + independent_final:", csv_summary_path)
print("\nCheckpoints saved under:")
print(" - Folds:", os.path.join(CKPT_ROOT, "fold*/"))
print(" - Final:", os.path.join(CKPT_ROOT, "final/"))
